# Day 7 — Solution: Stylized-Facts Audit (exemplar)

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    assets = {"SPY": get_prices("SPY", start="2005-01-01")["SPY"],
              "TLT": get_prices("TLT", start="2005-01-01")["TLT"],
              "XLE": get_prices("XLE", start="2005-01-01")["XLE"]}
else:
    _px = synthetic_prices(n_days=5000, n_assets=3, seed=36, corr=0.1)
    _px.columns = ["SPY", "TLT", "XLE"]
    assets = {c: _px[c] for c in _px.columns}
rets = {k: v.pct_change().dropna() for k, v in assets.items()}

## Part 1 — the audit table

In [ ]:
def kurt(x):
    z = (x - np.mean(x)) / np.std(x)
    return (z**4).mean() - 3

rows = []
for name, r in rets.items():
    n, T = len(r), len(r)
    band = 2 / np.sqrt(T)
    acf_r = [r.autocorr(k) for k in range(1, 11)]
    acf_a = [r.abs().autocorr(k) for k in range(1, 11)]
    weekly = r.resample("W").sum().dropna()
    monthly = r.resample("ME").sum().dropna()
    v21 = r.rolling(21).std()
    lev = r.corr(np.log(v21).diff().shift(-1))
    rows.append(dict(asset=name,
        kurt_d=f"{kurt(r.values):.1f}±{np.sqrt(24/n):.1f}",
        acf_r=f"max {max(abs(np.nan_to_num(acf_r))):.3f} vs band {band:.3f}",
        acf_abs1=f"{acf_a[0]:.2f}", acf_abs10=f"{np.nan_to_num(acf_a[9]):.2f}",
        kurt_wm=f"{kurt(weekly.values):.1f} / {kurt(monthly.values):.1f}",
        leverage=f"{lev:+.2f}"))
print(pd.DataFrame(rows).to_string(index=False))
# Fact 6 (volume): real data only — compute where volume available; else N/A

**Reference verdicts (real data; yours vary by window):** every asset:
fact 1 ✓ (κ 3–15, many SEs from 0), fact 2 ✓ (max |ACF(r)| inside or
grazing ±2/√T), fact 3 ✓ (ACF(|r|)₁ 0.1–0.25, decaying but positive at
lag 10 — weeks of memory), fact 4 ✓ (κ collapses daily→weekly→monthly,
e.g. 12→3→1), fact 5: SPY/XLE −0.1..−0.25 ✓, TLT ≈ 0 (the instructive
cell — leverage is an equity fact), fact 6 ✓ where volume is clean
(corr(|r|, volume) 0.3–0.6).

## Part 2 — exhibits

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# (a) QQ of fattest-tailed asset
name = max(rets, key=lambda k: kurt(rets[k].values))
r = rets[name]
z = np.sort((r - r.mean()) / r.std())
q = st.norm.ppf((np.arange(len(z)) + 0.5) / len(z))
axes[0].scatter(q, z, s=3); axes[0].plot([-4,4],[-4,4],"r--")
axes[0].set_title(f"QQ: {name} (kurtosis {kurt(r.values):.0f})")
# (b) ACF(|r|) of SPY
r = rets["SPY"]; T = len(r)
acf = [r.abs().autocorr(k) for k in range(1, 21)]
axes[1].plot(range(1, 21), acf, "o-")
axes[1].axhspan(-2/np.sqrt(T), 2/np.sqrt(T), color="gray", alpha=0.3)
axes[1].set_title("ACF(|r|), SPY")
# (c) kurtosis by frequency
freqs = {"daily": lambda x: x, "weekly": lambda x: x.resample("W").sum(),
         "monthly": lambda x: x.resample("ME").sum()}
for i, (nm, a) in enumerate(rets.items()):
    axes[2].bar(np.arange(3) + (i-1)*0.25,
                [kurt(f(a).dropna().values) for f in freqs.values()],
                width=0.25, label=nm)
axes[2].set_xticks(range(3)); axes[2].set_xticklabels(freqs)
axes[2].set_title("excess kurtosis by frequency"); axes[2].legend()
plt.tight_layout(); plt.show()

## Part 3 — the risk manager's paragraph (exemplar)

> "All three assets are fat-tailed at daily frequency (XLE worst:
> κ = 12 ± 0.4), disqualifying normal-VaR — the 99% normal VaR
> understates the empirical tail by ~3–5×. Volatility clusters
> (SPY |r| ACF₁ = 0.21, still 0.08 at lag 10), disqualifying iid
> simulation for drawdown studies — use blocks ≥ 63 days or GARCH.
> Aggregational Gaussianity holds (κ monthly < 1), so monthly-scale
> portfolio analytics are defensible. The leverage effect is an equity
> phenomenon (SPY/XLE −0.15/−0.20; TLT ≈ 0) — vol forecasts for bonds
> should not import equity asymmetry. Recommend: GARCH-t for SPY/XLE
> risk; EWMA acceptable for TLT at monthly horizon."

## Part 4 — the bias audit (exemplar)

- **Window:** halves disagree on magnitude, not direction (κ 2005–2014
  ≈ 14, 2015–2024 ≈ 8 — both huge; verdicts survive).
- **Multiplicity:** 18 cells at 2 SE → expect ~1 lucky "hit"; we have
  none borderline — all effects are 5+ SE. No luck-inflation concern.
- **Selection:** SPY/TLT/XLE are survivors of their own product
  classes and the most liquid names — their tails are *probably
  understated* relative to the asset class (illiquid names crash
  harder and vanish).
- **Data quality:** volume used only for fact 6; XLE volume has
  split-date artifacts around 2008 (vendor), noted; verdict for fact 6
  rests on SPY/TLT only.